In [1]:
# bbtransformer_analyzer.py 

import os
from pathlib import Path
from typing import List, Optional, Dict, Any
from bbtransformer import run_analysis


# 🔑 NEURO_X CONFIG: Matches weights_ICD_neuroX.pth and all downstream fine-tuned models
NEURO_XCONFIG = {
    'feature_dim': 414,
    'num_classes': 1,
    'embed_dim': 512,
    'num_heads': 8,
    'num_layers': 6,
    'n_kv_heads': 4,
    'embed_dim_age': 32,
    'embed_dim_ext': 16,
    'patch_size': 3,
    'patch_embed_ratio': 0.5,
    'temp_attn_hidden': 128,
    'dropout_input': 0.27,
    'dropout_patch': 0.27,
    'dropout_attn': 0.146,
    'dropout_ffn': 0.275,
    'dropout_classifier': 0.029,
    'dropout_temporal': 0.167,
    'stochastic_depth_rate': 0.1,
    'return_attn_weights': False,
}

TRAIN_PARAMS = {
    'epochs': 5000,
    'lr': 2.3157e-05,
    'weight_decay': 1.14e-06,
    'patience': 90
}


class BBTransformerAnalyzer:
    def __init__(
        self,
        base_dir: str,
        weights_dir: str = "weights",
        results_dir: str = "results",
        initial_weights: Optional[str] = None,
        min_composite: float = 0.60,
        max_trials_per_disorder: int = 50
    ):
        self.base_dir = Path(base_dir)
        self.weights_dir = Path(weights_dir)
        self.results_dir = Path(results_dir)
        self.weights_dir.mkdir(exist_ok=True)
        self.results_dir.mkdir(exist_ok=True)
        
        self.current_weights = initial_weights
        self.min_composite = min_composite
        self.max_trials = max_trials_per_disorder
        self.valid_models = []  # Track successful disorders

    def get_paths(self, disorder: str):
        return (
            self.base_dir / f"fmri_{disorder}.npz",
            self.base_dir / f"pheno_{disorder}.csv"
        )

    def is_valid(self, metrics: Dict[str, float]) -> bool:
        """Check if all core metrics meet clinical threshold."""
        return all(
            metrics.get(metric, 0) >= self.min_composite
            for metric in ['f1', 'roc_auc', 'accuracy', 'precision', 'recall']
        )

    def run_ordered_pipeline(self, disorders: List[str]) -> Dict[str, Any]:
        """
        Run transfer learning in strict biological order.
        Propagate weights from last VALID model, even if intermediate disorders fail.
        Uses NEURO_XCONFIG to ensure architectural fidelity across all transfers.
        """
        results_summary = {}

        for i, disorder in enumerate(disorders, 1):
            print(f"\n{'='*70}")
            print(f"PHASE {i}/{len(disorders)}: {disorder}")
            print(f"{'='*70}")

            data_path, pheno_path = self.get_paths(disorder)
            
            use_pretrained = self.current_weights is not None
            
            best_result = None
            for trial in range(self.max_trials):
                print(f"  Trial {trial+1}/{self.max_trials}...")

                try:
                    result = run_analysis(
                        model_config=NEURO_XCONFIG,      # ← FIXED: now uses compatible config
                        training_config=TRAIN_PARAMS,
                        target_column=disorder,
                        data_path=str(data_path),
                        pheno_path=str(pheno_path),
                        use_pretrained=use_pretrained,
                        pretrained_weight_file=self.current_weights,
                        compute_importance=False,
                        random_seed=42 + trial,
                        weights_dir=str(self.weights_dir)
                    )
                    
                    if self.is_valid(result['metrics']):
                        best_result = result
                        print(f"  ✅ VALID MODEL FOUND (Composite: {result['metrics']['f1']:.4f})")
                        break
                    else:
                        print(f"  ❌ Trial {trial+1} failed validity check")
                        
                except Exception as e:
                    print(f"  ❌ Trial {trial+1} crashed: {str(e)}")
                    continue

            # Save result regardless of validity
            results_summary[disorder] = {
                'valid': best_result is not None,
                'metrics': best_result['metrics'] if best_result else None,
                'weights_used': self.current_weights,
                'weights_saved': None
            }

            # Update weights ONLY if this disorder is valid
            if best_result is not None:
                weight_file = f"weights_{disorder}.pth"
                self.current_weights = str(self.weights_dir / weight_file)
                results_summary[disorder]['weights_saved'] = self.current_weights
                self.valid_models.append(disorder)
                print(f"  🔁 Propagating weights to next disorder")
            else:
                if self.valid_models:
                    print(f"  ⚠️ Keeping weights from last valid model: {self.valid_models[-1]}")
                else:
                    print(f"  🧼 No prior valid model—next disorder will train from scratch")
                    self.current_weights = None

        return results_summary

In [2]:
TARGET_CONDITIONS = [
    'NervousSystem_Cerebrovascular',
    'NervousSystem_Inflammatory_Infectious',
    'ICD_F31_Bipolar',
    'Psychopathology_Schizophrenia_Spectrum',
    'ICD_F32_Depressive_Episode',
    'NervousSystem_Sleep_Disorders',
    'Psychopathology_Substance_Use',
    'NervousSystem_Epilepsy_Status_Epilepticus',
    'NervousSystem_Parkinsons_Other_Movement',
    'NervousSystem_Multiple_Sclerosis_Other_Demyelinating',
]

In [3]:

analyzer = BBTransformerAnalyzer(
    base_dir='/mnt/movement/users/jaizor/xtra/data/fmri/chrt',
    weights_dir='/mnt/movement/users/jaizor/xtra/ΞΞ/__/weights',  # must contain weights_ICD_neuroX.pth
    initial_weights='weights_ICD_neuroX.pth',  # ← now valid
    min_composite=0.7,
    max_trials_per_disorder=15
)

results = analyzer.run_ordered_pipeline(TARGET_CONDITIONS)


PHASE 1/10: NervousSystem_Cerebrovascular
  Trial 1/15...
STEP 1: Loading Data for Target = 'NervousSystem_Cerebrovascular'
  fMRI: /mnt/movement/users/jaizor/xtra/data/fmri/chrt/fmri_NervousSystem_Cerebrovascular.npz
  Phenotype: /mnt/movement/users/jaizor/xtra/data/fmri/chrt/pheno_NervousSystem_Cerebrovascular.csv
Loaded phenotype: (592, 56)
Loaded fMRI: (592, 150, 414)
  Subjects: 592
  Timepoints: 150
  Brain regions: 414
  ROI labels: 414
All required columns present

STEP 2: Preparing Data Loaders
Cohort: 592 subjects (296 cases, 296 controls, 50.0% prevalence)
Splits → Train: 414, Val: 89, Test: 89

Dataset Meta
  target: NervousSystem_Cerebrovascular
  n_total: 592
  n_positive: 296
  prevalence: 0.5
  feature_dim: 414
  n_train: 414
  n_val: 89
  n_test: 89

STEP 3: Initializing BBTransformer
Model created on cuda with 25,878,528 parameters

STEP 3.5: Loading Pretrained Weights
  From: /mnt/movement/users/jaizor/xtra/ΞΞ/__/weights/weights_ICD_neuroX.pth
Attempting SAFE load (

Early stopping at epoch 160 (F1: 0.7767)
✓ Saved SAFE weights (weights_only=True compatible) to: weights/weights_NervousSystem_Cerebrovascular.pth

STEP 5: Test Set Evaluation



Performance Metrics:
  Accuracy:  0.8764
  Precision: 0.9714
  Recall:    0.7727
  F1 Score:  0.8608
  ROC-AUC:   0.9379

Confusion Matrix:
[[44  1]
 [10 34]]

[INFO] Skipping permutation importance (compute_importance=False).
Results saved to JSON: results/NervousSystem_Cerebrovascular_results.json

TRAINING & EVALUATION COMPLETE
Target: NervousSystem_Cerebrovascular
  ✅ VALID MODEL FOUND (Composite: 0.8608)
  🔁 Propagating weights to next disorder

PHASE 2/10: NervousSystem_Inflammatory_Infectious
  Trial 1/15...
STEP 1: Loading Data for Target = 'NervousSystem_Inflammatory_Infectious'
  fMRI: /mnt/movement/users/jaizor/xtra/data/fmri/chrt/fmri_NervousSystem_Inflammatory_Infectious.npz
  Phenotype: /mnt/movement/users/jaizor/xtra/data/fmri/chrt/pheno_NervousSystem_Inflammatory_Infectious.csv
Loaded phenotype: (92, 56)
Loaded fMRI: (92, 150, 414)
  Subjects: 92
  Timepoints: 150
  Brain regions: 414
  ROI labels: 414
All required columns present

STEP 2: Preparing Data Loaders
Cohort

Early stopping at epoch 91 (F1: 1.0000)
✓ Saved SAFE weights (weights_only=True compatible) to: weights/weights_NervousSystem_Inflammatory_Infectious.pth

STEP 5: Test Set Evaluation



Performance Metrics:
  Accuracy:  1.0000
  Precision: 1.0000
  Recall:    1.0000
  F1 Score:  1.0000
  ROC-AUC:   1.0000

Confusion Matrix:
[[7 0]
 [0 7]]

[INFO] Skipping permutation importance (compute_importance=False).
Results saved to JSON: results/NervousSystem_Inflammatory_Infectious_results.json

TRAINING & EVALUATION COMPLETE
Target: NervousSystem_Inflammatory_Infectious
  ✅ VALID MODEL FOUND (Composite: 1.0000)
  🔁 Propagating weights to next disorder

PHASE 3/10: ICD_F31_Bipolar
  Trial 1/15...
STEP 1: Loading Data for Target = 'ICD_F31_Bipolar'
  fMRI: /mnt/movement/users/jaizor/xtra/data/fmri/chrt/fmri_ICD_F31_Bipolar.npz
  Phenotype: /mnt/movement/users/jaizor/xtra/data/fmri/chrt/pheno_ICD_F31_Bipolar.csv
Loaded phenotype: (110, 56)
Loaded fMRI: (110, 150, 414)
  Subjects: 110
  Timepoints: 150
  Brain regions: 414
  ROI labels: 414
All required columns present

STEP 2: Preparing Data Loaders
Cohort: 110 subjects (55 cases, 55 controls, 50.0% prevalence)
Splits → Train: 

...


Performance Metrics:
  Accuracy:  0.9423
  Precision: 1.0000
  Recall:    0.8846
  F1 Score:  0.9388
  ROC-AUC:   1.0000

Confusion Matrix:
[[26  0]
 [ 3 23]]

[INFO] Skipping permutation importance (compute_importance=False).
Results saved to JSON: results/NervousSystem_Epilepsy_Status_Epilepticus_results.json

TRAINING & EVALUATION COMPLETE
Target: NervousSystem_Epilepsy_Status_Epilepticus
  ✅ VALID MODEL FOUND (Composite: 0.9388)
  🔁 Propagating weights to next disorder

PHASE 9/10: NervousSystem_Parkinsons_Other_Movement
  Trial 1/15...
STEP 1: Loading Data for Target = 'NervousSystem_Parkinsons_Other_Movement'
  fMRI: /mnt/movement/users/jaizor/xtra/data/fmri/chrt/fmri_NervousSystem_Parkinsons_Other_Movement.npz
  Phenotype: /mnt/movement/users/jaizor/xtra/data/fmri/chrt/pheno_NervousSystem_Parkinsons_Other_Movement.csv
Loaded phenotype: (416, 56)
Loaded fMRI: (416, 150, 414)
  Subjects: 416
  Timepoints: 150
  Brain regions: 414
  ROI labels: 414
All required columns present

ST

Early stopping at epoch 204 (F1: 0.8406)
✓ Saved SAFE weights (weights_only=True compatible) to: weights/weights_NervousSystem_Parkinsons_Other_Movement.pth

STEP 5: Test Set Evaluation



Performance Metrics:
  Accuracy:  0.9683
  Precision: 0.9677
  Recall:    0.9677
  F1 Score:  0.9677
  ROC-AUC:   0.9869

Confusion Matrix:
[[31  1]
 [ 1 30]]

[INFO] Skipping permutation importance (compute_importance=False).
Results saved to JSON: results/NervousSystem_Parkinsons_Other_Movement_results.json

TRAINING & EVALUATION COMPLETE
Target: NervousSystem_Parkinsons_Other_Movement
  ✅ VALID MODEL FOUND (Composite: 0.9677)
  🔁 Propagating weights to next disorder

PHASE 10/10: NervousSystem_Multiple_Sclerosis_Other_Demyelinating
  Trial 1/15...
STEP 1: Loading Data for Target = 'NervousSystem_Multiple_Sclerosis_Other_Demyelinating'
  fMRI: /mnt/movement/users/jaizor/xtra/data/fmri/chrt/fmri_NervousSystem_Multiple_Sclerosis_Other_Demyelinating.npz
  Phenotype: /mnt/movement/users/jaizor/xtra/data/fmri/chrt/pheno_NervousSystem_Multiple_Sclerosis_Other_Demyelinating.csv
Loaded phenotype: (164, 56)
Loaded fMRI: (164, 150, 414)
  Subjects: 164
  Timepoints: 150
  Brain regions: 414
 

Early stopping at epoch 91 (F1: 0.9600)
✓ Saved SAFE weights (weights_only=True compatible) to: weights/weights_NervousSystem_Multiple_Sclerosis_Other_Demyelinating.pth

STEP 5: Test Set Evaluation



Performance Metrics:
  Accuracy:  1.0000
  Precision: 1.0000
  Recall:    1.0000
  F1 Score:  1.0000
  ROC-AUC:   1.0000

Confusion Matrix:
[[13  0]
 [ 0 12]]

[INFO] Skipping permutation importance (compute_importance=False).
Results saved to JSON: results/NervousSystem_Multiple_Sclerosis_Other_Demyelinating_results.json

TRAINING & EVALUATION COMPLETE
Target: NervousSystem_Multiple_Sclerosis_Other_Demyelinating
  ✅ VALID MODEL FOUND (Composite: 1.0000)
  🔁 Propagating weights to next disorder
